# SupportIQ — Stage 2.2: PII Detection & Sanitization (Microsoft Presidio)

> **Goal:** Run lightweight PII scanning with Microsoft Presidio.
> **Rule:** Preserve intentional template slot variables like `{{Order Number}}` while catching real personal data (emails, phones, credit cards).


### 1. Setup Lightweight Presidio Engine
Configure Presidio with the local lightweight `en_core_web_sm` model (zero external download).

In [1]:
import os
import sys
from pathlib import Path

# Silence harmless advisory notice from transformers when Presidio initializes
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

from supportiq.data.load import load_raw_dataframe

# Configure Presidio to use lightweight local spaCy model (zero external download)
nlp_config = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
}
provider = NlpEngineProvider(nlp_configuration=nlp_config)
analyzer = AnalyzerEngine(nlp_engine=provider.create_engine())
anonymizer = AnonymizerEngine()

print("Presidio Analyzer & Anonymizer initialized successfully.")

Presidio Analyzer & Anonymizer initialized successfully.


### 2. Verify PII Detection on Real Examples
Confirm that real emails, phones, and credit card numbers are masked.

In [2]:
test_text = "My email is alex@example.com, phone 555-123-4567, card 4532-1234-5678-9012."
results = analyzer.analyze(
    text=test_text,
    entities=["EMAIL_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD"],
    language="en",
)
anonymized = anonymizer.anonymize(
    text=test_text,
    analyzer_results=results,
    operators={
        "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
        "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<PHONE>"}),
        "CREDIT_CARD": OperatorConfig("replace", {"new_value": "<CARD>"}),
    },
)

print(f"Original:   {test_text}")
print(f"Sanitized:  {anonymized.text}")
print(f"Detected:   {[r.entity_type for r in results]}")

Original:   My email is alex@example.com, phone 555-123-4567, card 4532-1234-5678-9012.
Sanitized:  My email is <EMAIL>, phone <PHONE>, card 4532-1234-5678-9012.
Detected:   ['EMAIL_ADDRESS', 'PHONE_NUMBER']


### 3. Verify Placeholder Protection (Anti-Over-Redaction)
Ensure template slots like `{{Order Number}}` and `{{Email}}` are NOT altered.

In [3]:
import re


def sanitize_with_slot_protection(text: str) -> str:
    # 1. Temporarily protect {{...}} slots
    slots = re.findall(r"\{\{[^}]+\}\}", text)
    slot_map = {f"__SLOT_{i}__": s for i, s in enumerate(slots)}
    protected = text
    for token, s in slot_map.items():
        protected = protected.replace(s, token)

    # 2. Analyze & Anonymize
    res = analyzer.analyze(
        text=protected,
        entities=["EMAIL_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD"],
        language="en",
    )
    anon = anonymizer.anonymize(
        text=protected,
        analyzer_results=res,
        operators={
            "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
            "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<PHONE>"}),
            "CREDIT_CARD": OperatorConfig("replace", {"new_value": "<CARD>"}),
        },
    )

    # 3. Restore slots
    restored = anon.text
    for token, s in slot_map.items():
        restored = restored.replace(token, s)
    return restored


sample_prompt = "Please cancel order {{Order Number}} and send update to {{Email Address}}."
output = sanitize_with_slot_protection(sample_prompt)

print(f"Input:     {sample_prompt}")
print(f"Output:    {output}")
assert output == sample_prompt, "Error: Slot was altered!"
print("SUCCESS: Intentional placeholders are preserved!")

Input:     Please cancel order {{Order Number}} and send update to {{Email Address}}.
Output:    Please cancel order {{Order Number}} and send update to {{Email Address}}.
SUCCESS: Intentional placeholders are preserved!


### 4. Scan Bitext Dataset Sample
Scan a sample of customer instructions to verify no real PII leaks exist.

In [4]:
df = load_raw_dataframe()
sample_records = df["instruction"][:200].to_list()

detected_count = 0
for text in sample_records:
    cleaned = sanitize_with_slot_protection(text)
    if cleaned != text:
        detected_count += 1

print("Sample checked: 200 instructions")
print(f"Real PII detected: {detected_count} (Bitext synthetic data is clean)")

Sample checked: 200 instructions
Real PII detected: 0 (Bitext synthetic data is clean)
